# QAI Training Evaluation

This notebook loads QAI training artifacts and builds a visual review of model behavior across epochs.

It expects a run directory with files such as:
- `metrics.json`
- `epoch_metrics.json` or `tables/epoch_metrics.csv`
- `artifacts/training_hyperparameters.json`
- `artifacts/data_summary.json`
- optional `tables/predictions.csv`


In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

try:
    import matplotlib.pyplot as plt
except ImportError as exc:
    raise ImportError(
        "matplotlib is required to run this notebook. Install it in the environment first."
    ) from exc

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)


In [ ]:
def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src").exists():
            return candidate
    raise FileNotFoundError("Could not locate the project root from the current working directory.")


def load_json(path: Path):
    if not path.exists():
        return None
    return json.loads(path.read_text())


def resolve_run_dir(run_dir: str | Path | None = None) -> Path:
    repo_root = find_repo_root()
    if run_dir is None:
        return repo_root
    run_dir = Path(run_dir)
    if run_dir.is_absolute():
        return run_dir
    return (repo_root / run_dir).resolve()


RUN_DIR = resolve_run_dir()
RUN_DIR


In [ ]:
def load_epoch_frame(run_dir: Path) -> pd.DataFrame:
    csv_path = run_dir / "tables" / "epoch_metrics.csv"
    json_path = run_dir / "epoch_metrics.json"
    if csv_path.exists():
        frame = pd.read_csv(csv_path)
    elif json_path.exists():
        frame = pd.DataFrame(load_json(json_path))
    else:
        raise FileNotFoundError("Missing epoch history. Expected tables/epoch_metrics.csv or epoch_metrics.json")
    if "epoch" in frame.columns:
        frame = frame.sort_values("epoch").reset_index(drop=True)
    return frame


epoch_df = load_epoch_frame(RUN_DIR)
final_metrics = load_json(RUN_DIR / "metrics.json")
best_metrics = load_json(RUN_DIR / "artifacts" / "best_metrics.json")
best_checkpoint_summary = load_json(RUN_DIR / "artifacts" / "best_checkpoint_summary.json")
hyperparams = load_json(RUN_DIR / "artifacts" / "training_hyperparameters.json")
data_summary = pd.DataFrame(load_json(RUN_DIR / "artifacts" / "data_summary.json") or [])
predictions_path = RUN_DIR / "tables" / "predictions.csv"
predictions_df = pd.read_csv(predictions_path) if predictions_path.exists() else None

display(Markdown(f"## Loaded Run: `{RUN_DIR}`"))
display(Markdown(f"Epochs loaded: **{len(epoch_df)}**"))
epoch_df.head()


In [ ]:
summary_rows = []
if hyperparams:
    summary_rows.append({"section": "training_hyperparameters", **hyperparams})
if final_metrics:
    summary_rows.append({"section": "final_metrics", **final_metrics})
if best_metrics:
    summary_rows.append({"section": "best_metrics", **best_metrics})

display(Markdown("## Run Tables"))
if not data_summary.empty:
    display(Markdown("### Data Summary"))
    display(data_summary)

if summary_rows:
    display(Markdown("### Key Run Metadata"))
    display(pd.DataFrame(summary_rows))


In [ ]:
best_epoch = None
if "val_loss" in epoch_df.columns and not epoch_df["val_loss"].isna().all():
    best_epoch = epoch_df.loc[epoch_df["val_loss"].idxmin(), "epoch"]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(epoch_df["epoch"], epoch_df["train_loss"], marker="o", label="train")
axes[0].plot(epoch_df["epoch"], epoch_df["val_loss"], marker="o", label="val")
axes[0].set_title("Loss by Epoch")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()

axes[1].plot(epoch_df["epoch"], epoch_df["train_price_mae"], marker="o", label="train")
axes[1].plot(epoch_df["epoch"], epoch_df["val_price_mae"], marker="o", label="val")
axes[1].set_title("Price MAE by Epoch")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("MAE")
axes[1].legend()

axes[2].plot(epoch_df["epoch"], epoch_df["train_class_accuracy"], marker="o", label="train")
axes[2].plot(epoch_df["epoch"], epoch_df["val_class_accuracy"], marker="o", label="val")
axes[2].set_title("Classification Accuracy by Epoch")
axes[2].set_xlabel("Epoch")
axes[2].set_ylabel("Accuracy")
axes[2].legend()

if best_epoch is not None:
    for ax in axes:
        ax.axvline(best_epoch, color="tab:red", linestyle="--", alpha=0.6)

plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].plot(epoch_df["epoch"], epoch_df["train_regression_loss"], marker="o", label="train regression")
axes[0].plot(epoch_df["epoch"], epoch_df["val_regression_loss"], marker="o", label="val regression")
axes[0].plot(epoch_df["epoch"], epoch_df["train_classification_loss"], marker="o", label="train classification")
axes[0].plot(epoch_df["epoch"], epoch_df["val_classification_loss"], marker="o", label="val classification")
axes[0].set_title("Loss Components")
axes[0].set_xlabel("Epoch")
axes[0].legend()

epoch_df["loss_gap"] = epoch_df["val_loss"] - epoch_df["train_loss"]
epoch_df["mae_gap"] = epoch_df["val_price_mae"] - epoch_df["train_price_mae"]
epoch_df["accuracy_gap"] = epoch_df["train_class_accuracy"] - epoch_df["val_class_accuracy"]

axes[1].plot(epoch_df["epoch"], epoch_df["loss_gap"], marker="o", label="val_loss - train_loss")
axes[1].plot(epoch_df["epoch"], epoch_df["mae_gap"], marker="o", label="val_mae - train_mae")
axes[1].plot(epoch_df["epoch"], epoch_df["accuracy_gap"], marker="o", label="train_acc - val_acc")
axes[1].axhline(0.0, color="black", linewidth=1)
axes[1].set_title("Generalization Gaps")
axes[1].set_xlabel("Epoch")
axes[1].legend()

plt.tight_layout()
plt.show()


In [ ]:
train_horizon_cols = [column for column in epoch_df.columns if column.startswith("train_class_accuracy_h")]
val_horizon_cols = [column for column in epoch_df.columns if column.startswith("val_class_accuracy_h")]

fig, axes = plt.subplots(1, 2, figsize=(18, 5), sharey=True)
for column in train_horizon_cols:
    axes[0].plot(epoch_df["epoch"], epoch_df[column], marker="o", label=column.replace("train_", ""))
axes[0].set_title("Train Accuracy by Horizon")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Accuracy")
axes[0].legend()

for column in val_horizon_cols:
    axes[1].plot(epoch_df["epoch"], epoch_df[column], marker="o", label=column.replace("val_", ""))
axes[1].set_title("Validation Accuracy by Horizon")
axes[1].set_xlabel("Epoch")
axes[1].legend()

plt.tight_layout()
plt.show()


In [ ]:
eval_df = epoch_df.copy()
for column in [
    "train_loss",
    "val_loss",
    "train_price_mae",
    "val_price_mae",
    "train_class_accuracy",
    "val_class_accuracy",
]:
    if column in eval_df.columns:
        eval_df[f"delta_{column}"] = eval_df[column].diff()

display(Markdown("## Epoch-over-Epoch Changes"))
display(eval_df[[column for column in eval_df.columns if column.startswith(("epoch", "delta_"))]].tail(10))

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
axes[0].bar(epoch_df["epoch"], epoch_df["val_loss"] - epoch_df["val_loss"].min())
axes[0].set_title("Distance From Best Validation Loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss Above Best")

axes[1].bar(epoch_df["epoch"], epoch_df["val_class_accuracy"] - epoch_df["val_class_accuracy"].min())
axes[1].set_title("Validation Accuracy Lift Over Worst Epoch")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy Gain")

plt.tight_layout()
plt.show()


In [ ]:
if predictions_df is None:
    display(Markdown("## Prediction-Level Evaluation"))
    display(Markdown("`tables/predictions.csv` was not found. Run the evaluator first if you want prediction-level plots."))
else:
    prediction_cols = list(predictions_df.columns)
    display(Markdown("## Prediction-Level Evaluation"))
    display(predictions_df.head())

    if {"prediction", "target"}.issubset(predictions_df.columns):
        preds = pd.to_numeric(predictions_df["prediction"], errors="coerce")
        targets = pd.to_numeric(predictions_df["target"], errors="coerce")
        valid = pd.DataFrame({"prediction": preds, "target": targets}).dropna()
        valid["residual"] = valid["prediction"] - valid["target"]

        eval_summary = pd.DataFrame([
            {
                "mae": mean_absolute_error(valid["target"], valid["prediction"]),
                "rmse": mean_squared_error(valid["target"], valid["prediction"], squared=False),
                "r2": r2_score(valid["target"], valid["prediction"]),
                "mean_residual": valid["residual"].mean(),
            }
        ])
        display(eval_summary)

        fig, axes = plt.subplots(1, 3, figsize=(18, 5))
        axes[0].scatter(valid["target"], valid["prediction"], alpha=0.6)
        diagonal_min = min(valid["target"].min(), valid["prediction"].min())
        diagonal_max = max(valid["target"].max(), valid["prediction"].max())
        axes[0].plot([diagonal_min, diagonal_max], [diagonal_min, diagonal_max], color="tab:red", linestyle="--")
        axes[0].set_title("Prediction vs Target")
        axes[0].set_xlabel("Target")
        axes[0].set_ylabel("Prediction")

        axes[1].hist(valid["residual"], bins=30, color="tab:blue", alpha=0.8)
        axes[1].set_title("Residual Distribution")
        axes[1].set_xlabel("Prediction - Target")

        axes[2].scatter(valid["prediction"], valid["residual"], alpha=0.6)
        axes[2].axhline(0.0, color="black", linewidth=1)
        axes[2].set_title("Residuals vs Prediction")
        axes[2].set_xlabel("Prediction")
        axes[2].set_ylabel("Residual")

        plt.tight_layout()
        plt.show()
    else:
        display(Markdown(f"Predictions file columns are `{prediction_cols}`. Add notebook-specific parsing if evaluator output changes."))


In [ ]:
display(Markdown("## Best Epoch Snapshot"))
if best_checkpoint_summary is not None:
    display(pd.DataFrame([best_checkpoint_summary]))
elif best_metrics is not None:
    best_epoch_row = pd.DataFrame([best_metrics])
    display(best_epoch_row)
else:
    best_epoch_row = epoch_df.loc[[epoch_df["val_loss"].idxmin()]] if "val_loss" in epoch_df.columns else epoch_df.tail(1)
    display(best_epoch_row)

display(Markdown("## Last 10 Epochs"))
display(epoch_df.tail(10))
